# 🧹 Guía completa de Preprocesamiento de Datos en Python

> **Notebook de referencia.** Recopila las técnicas más habituales de limpieza y
> preparación de datos con `pandas`, `numpy` y `scikit-learn`, con ejemplos de
> código comentados y explicación de cada sección.
>
> Usamos el dataset **Titanic** como hilo conductor porque mezcla variables
> numéricas, categóricas, texto, fechas implícitas y valores nulos — ideal para
> ilustrar casi cualquier técnica.

## 📑 Índice

1. [Librerías](#1)
2. [Carga de datos](#2)
3. [Exploración inicial (EDA básico)](#3)
4. [Análisis de valores nulos](#4)
5. [Tratamiento de valores nulos](#5)
6. [Duplicados](#6)
7. [Tipos de datos y conversiones](#7)
8. [Limpieza de texto / strings](#8)
9. [Detección y tratamiento de outliers](#9)
10. [Codificación de variables categóricas](#10)
11. [Escalado y normalización](#11)
12. [Transformación de distribuciones](#12)
13. [Feature engineering (creación de variables)](#13)
14. [Selección de características](#14)
15. [Datos desbalanceados](#15)
16. [División train / test](#16)
17. [Pipelines y ColumnTransformer](#17)
18. [Guardado del dataset y del pipeline](#18)
19. [Checklist final](#19)

> 💡 **Cómo usar este notebook:** las celdas con datos reales del Titanic se pueden
> ejecutar tal cual. Las celdas marcadas con `# ILUSTRATIVO` muestran la sintaxis
> de una técnica que quizá no aplique a este dataset concreto, pero que conviene
> tener a mano como plantilla.


<a id="1"></a>
## 1. Librerías

El stack estándar de preprocesamiento:

- **pandas** → manipulación de tablas (DataFrames).
- **numpy** → operaciones numéricas y de arrays.
- **matplotlib / seaborn** → visualización.
- **scikit-learn** → imputación, codificación, escalado y *pipelines*.

> Para instalar lo que falte:
> `%pip install pandas numpy matplotlib seaborn scikit-learn`


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Opciones cómodas de visualización
pd.set_option('display.max_columns', None)   # mostrar todas las columnas
pd.set_option('display.width', 120)
sns.set_theme(style='whitegrid')

print('pandas', pd.__version__)
print('numpy', np.__version__)

<a id="2"></a>
## 2. Carga de datos

`pandas` lee prácticamente cualquier formato. El más común es CSV, pero conviene
conocer las alternativas y los parámetros más útiles de `read_csv`.


In [ ]:
# --- CSV (lo más habitual) ---
df_train = pd.read_csv('../data/train.csv')
df_test  = pd.read_csv('../data/test.csv')

df_train.head(3)

In [ ]:
# ILUSTRATIVO — parámetros útiles de read_csv
# pd.read_csv(
#     'archivo.csv',
#     sep=';',                 # separador (en España suele ser ';')
#     decimal=',',             # coma decimal
#     encoding='utf-8',        # o 'latin-1' si hay acentos raros
#     na_values=['', 'NA', '?', 'null'],   # qué tratar como NaN
#     parse_dates=['fecha'],   # parsear fechas al cargar
#     dtype={'codigo': str},   # forzar tipo de una columna
#     usecols=['A', 'B'],      # leer solo algunas columnas
#     nrows=1000,              # leer solo N filas (datasets enormes)
#     thousands='.',           # separador de miles
# )

# --- Otros formatos ---
# pd.read_excel('archivo.xlsx', sheet_name='Hoja1')
# pd.read_json('archivo.json')
# pd.read_parquet('archivo.parquet')      # eficiente para datasets grandes
# pd.read_sql('SELECT * FROM tabla', conexion)
# pd.read_clipboard()                     # pega lo que tengas copiado

<a id="3"></a>
## 3. Exploración inicial (EDA básico)

Antes de tocar nada, hay que **entender** los datos: dimensiones, tipos,
estadísticos y cardinalidad de cada variable. Este paso guía todas las decisiones
posteriores.


In [ ]:
print('Dimensiones (filas, columnas):', df_train.shape)
print('\nColumnas:', list(df_train.columns))

In [ ]:
# .info() → tipos de dato y nº de no-nulos por columna (clave para ver nulos)
df_train.info()

In [ ]:
# .describe() → estadísticos de las columnas numéricas
df_train.describe()

In [ ]:
# .describe(include='object') → estadísticos de las categóricas/texto
df_train.describe(include='object')

In [ ]:
# Cardinalidad: cuántos valores únicos tiene cada columna
df_train.nunique().sort_values(ascending=False)

In [ ]:
# Distribución de una variable categórica concreta
print(df_train['Sex'].value_counts())
print()
print(df_train['Embarked'].value_counts(normalize=True))   # en proporción

<a id="4"></a>
## 4. Análisis de valores nulos

Identificar **dónde** y **cuántos** nulos hay es imprescindible para decidir si se
eliminan, se imputan o se deja la columna fuera.


In [ ]:
# Recuento y porcentaje de nulos por columna
nulos = pd.DataFrame({
    'n_nulos': df_train.isnull().sum(),
    'pct_nulos': (df_train.isnull().sum() / len(df_train) * 100).round(2)
})
nulos[nulos['n_nulos'] > 0].sort_values('pct_nulos', ascending=False)

In [ ]:
# Mapa de calor de nulos: visualiza el patrón de datos faltantes
plt.figure(figsize=(10, 4))
sns.heatmap(df_train.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Mapa de valores nulos (amarillo = nulo)')
plt.show()

In [ ]:
# ILUSTRATIVO — la librería `missingno` da visualizaciones de nulos muy claras
# %pip install missingno
# import missingno as msno
# msno.matrix(df_train)     # patrón de nulos
# msno.bar(df_train)        # barras de completitud
# msno.heatmap(df_train)    # correlación de nulos entre columnas

<a id="5"></a>
## 5. Tratamiento de valores nulos

Estrategias, de menos a más sofisticadas:

| Estrategia | Cuándo usarla |
|---|---|
| **Eliminar columna** | La columna tiene demasiados nulos (p.ej. >70-80%) y no es recuperable |
| **Eliminar filas** | Hay muy pocos nulos y perder esas filas no afecta |
| **Imputar (media/mediana/moda)** | Imputación clásica y rápida |
| **ffill / bfill / interpolate** | Series temporales u ordenadas |
| **Imputación avanzada (KNN, iterativa)** | Cuando se quiere aprovechar la relación entre variables |

> ⚠️ **Regla de oro:** los parámetros de imputación (mediana, moda…) deben
> calcularse **solo con el train** y aplicarse al test, para no filtrar
> información (*data leakage*).


In [ ]:
# Trabajamos sobre una copia para no alterar el original
df = df_train.copy()

In [ ]:
# --- 5.1 Eliminar columna con demasiados nulos ---
# 'Cabin' tiene ~77% de nulos → la eliminamos
df = df.drop(columns=['Cabin'])
print('Cabin eliminada. Columnas restantes:', list(df.columns))

In [ ]:
# --- 5.2 Eliminar filas con nulos ---
# dropna elimina cualquier fila que tenga AL MENOS un nulo
# df.dropna()                          # todas las filas con algún nulo
# df.dropna(subset=['Embarked'])       # solo si 'Embarked' es nulo
# df.dropna(thresh=10)                 # conservar filas con >=10 valores no nulos

In [ ]:
# --- 5.3 Imputación clásica con pandas ---
# Numérica → mediana (robusta frente a outliers; mejor que la media aquí)
df['Age'] = df['Age'].fillna(df['Age'].median())

# Categórica → moda (el valor más frecuente)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

print('Nulos restantes:')
print(df.isnull().sum())

In [ ]:
# ILUSTRATIVO — imputación en datos ordenados / series temporales
# df['col'].ffill()                 # arrastra hacia delante el último valor válido
# df['col'].bfill()                 # arrastra hacia atrás el siguiente válido
# df['col'].interpolate()           # interpola linealmente entre valores
# df['col'].interpolate(method='time')   # interpolación según el índice temporal

In [ ]:
# --- 5.4 Imputación con scikit-learn (SimpleImputer) ---
# Ventaja: se integra en pipelines y "aprende" del train para aplicarse al test.
from sklearn.impute import SimpleImputer

imp_num = SimpleImputer(strategy='median')      # 'mean', 'median', 'most_frequent', 'constant'
df_train[['Age']] = imp_num.fit_transform(df_train[['Age']])
print('Mediana aprendida por el imputer:', imp_num.statistics_)

In [ ]:
# ILUSTRATIVO — imputación avanzada
# KNNImputer: imputa según los k vecinos más parecidos
# from sklearn.impute import KNNImputer
# knn = KNNImputer(n_neighbors=5)
# df_num_imputado = knn.fit_transform(df[['Age', 'Fare', 'Pclass']])

# IterativeImputer: modela cada columna en función de las demás (MICE)
# from sklearn.experimental import enable_iterative_imputer  # noqa
# from sklearn.impute import IterativeImputer
# it = IterativeImputer(random_state=0)
# df_imputado = it.fit_transform(df[['Age', 'Fare', 'Pclass']])

<a id="6"></a>
## 6. Duplicados

Las filas duplicadas sesgan el modelo (dan más peso a esas observaciones) e inflan
las métricas. Hay que detectarlas y, normalmente, eliminarlas.


In [ ]:
# ¿Cuántas filas están completamente duplicadas?
print('Duplicados exactos:', df.duplicated().sum())

# Duplicados según un subconjunto de columnas (p.ej. mismo nombre)
print('Nombres duplicados:', df.duplicated(subset=['Name']).sum())

In [ ]:
# Eliminar duplicados
# df = df.drop_duplicates()                       # filas idénticas
# df = df.drop_duplicates(subset=['Name'],        # según columnas
#                         keep='first')           # 'first', 'last' o False (eliminar todas)

<a id="7"></a>
## 7. Tipos de datos y conversiones

Un tipo incorrecto rompe cálculos y modelos. Conviene asegurarse de que cada
columna tiene el tipo adecuado: numérico, fecha, categórico o texto.


In [ ]:
# Conversión explícita de tipos
df['Pclass'] = df['Pclass'].astype('category')        # categórica ordenable
df['Survived'] = df['Survived'].astype('int64')

# astype('category') ahorra memoria y acelera operaciones en variables con
# pocos valores únicos.
df.dtypes

In [ ]:
# ILUSTRATIVO — conversiones robustas
# to_numeric con errors='coerce' → lo que no se pueda convertir pasa a NaN
df['Fare'] = pd.to_numeric(df['Fare'], errors='coerce')

# to_datetime → texto a fecha
# df['fecha'] = pd.to_datetime(df['fecha'], format='%Y-%m-%d', errors='coerce')

# Int64 (con I mayúscula) admite enteros CON nulos, a diferencia de int64
# df['col'] = df['col'].astype('Int64')

<a id="8"></a>
## 8. Limpieza de texto / strings

El texto suele venir con espacios, mayúsculas inconsistentes o patrones que se
pueden extraer. El *accessor* `.str` de pandas aplica operaciones de cadena a toda
la columna de forma vectorizada.


In [ ]:
# Operaciones de cadena habituales (.str)
ejemplo = df['Name'].head()
print(ejemplo.str.lower().tolist()[:2])        # minúsculas
print(ejemplo.str.strip().tolist()[:2])        # quitar espacios sobrantes
print(ejemplo.str.len().tolist())              # longitud

In [ ]:
# Extraer información con expresiones regulares.
# Ejemplo clásico del Titanic: el TÍTULO está entre la coma y el punto del nombre.
df['Titulo'] = df['Name'].str.extract(r',\s*([^\.]+)\.')
df['Titulo'].value_counts()

In [ ]:
# ILUSTRATIVO — otras operaciones de texto
# df['col'].str.replace('antiguo', 'nuevo', regex=False)
# df['col'].str.contains('patron', case=False, na=False)   # filtro booleano
# df['col'].str.split(',', expand=True)                    # dividir en columnas
# df['col'].str.replace(r'[^0-9]', '', regex=True)         # dejar solo dígitos

<a id="9"></a>
## 9. Detección y tratamiento de outliers

Los valores atípicos pueden ser errores o casos reales extremos. Dos métodos
clásicos para detectarlos:

- **Rango intercuartílico (IQR):** atípico si está fuera de `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`.
- **Z-score:** atípico si `|z| > 3` (a más de 3 desviaciones típicas de la media).


In [ ]:
# Visualización rápida con boxplot
plt.figure(figsize=(8, 3))
sns.boxplot(x=df['Fare'])
plt.title('Boxplot de Fare (los puntos a la derecha son outliers)')
plt.show()

In [ ]:
# --- Método IQR ---
Q1 = df['Fare'].quantile(0.25)
Q3 = df['Fare'].quantile(0.75)
IQR = Q3 - Q1
lim_inf = Q1 - 1.5 * IQR
lim_sup = Q3 + 1.5 * IQR

outliers = df[(df['Fare'] < lim_inf) | (df['Fare'] > lim_sup)]
print(f'Límites válidos: [{lim_inf:.2f}, {lim_sup:.2f}]')
print(f'Nº de outliers en Fare: {len(outliers)}')

In [ ]:
# --- Tratamiento ---
# Opción A: ELIMINAR las filas outlier
# df = df[(df['Fare'] >= lim_inf) & (df['Fare'] <= lim_sup)]

# Opción B: RECORTAR (winsorizing) — fijar al límite en lugar de eliminar.
# Conserva todas las filas y suele ser preferible.
df['Fare'] = df['Fare'].clip(lower=lim_inf, upper=lim_sup)
print('Fare recortada al rango válido.')

In [ ]:
# ILUSTRATIVO — método Z-score
# from scipy import stats
# z = np.abs(stats.zscore(df['Fare']))
# df_sin_outliers = df[z < 3]

<a id="10"></a>
## 10. Codificación de variables categóricas

Los modelos de ML trabajan con números, así que hay que **codificar** las
categóricas. Según el tipo de variable:

| Técnica | Uso |
|---|---|
| **Mapeo manual / `map`** | Pocas categorías con orden claro (binarias) |
| **One-Hot (`get_dummies`)** | Categóricas **nominales** (sin orden): Sex, Embarked |
| **Ordinal / Label Encoding** | Categóricas **ordinales** (con orden): talla S<M<L |
| **Target / Frequency Encoding** | Alta cardinalidad (muchas categorías) |


In [ ]:
# --- 10.1 Mapeo manual (ideal para binarias) ---
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df['Sex'].value_counts()

In [ ]:
# --- 10.2 One-Hot Encoding con pandas (nominales) ---
# Crea una columna 0/1 por categoría. drop_first evita la multicolinealidad.
df = pd.get_dummies(df, columns=['Embarked'], prefix='Emb', drop_first=True)
[c for c in df.columns if c.startswith('Emb')]

In [ ]:
# ILUSTRATIVO — codificadores de scikit-learn (se integran en pipelines)
# from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder

# OneHotEncoder (para pipelines; handle_unknown evita errores con categorías nuevas)
# ohe = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
# ohe.fit_transform(df[['Embarked']])

# OrdinalEncoder (categóricas CON orden)
# orden = [['bajo', 'medio', 'alto']]
# OrdinalEncoder(categories=orden).fit_transform(df[['nivel']])

# LabelEncoder — SOLO para la variable objetivo (y), no para features
# LabelEncoder().fit_transform(y)

In [ ]:
# ILUSTRATIVO — Frequency encoding (alta cardinalidad, p.ej. 'Ticket')
# freq = df['Ticket'].value_counts(normalize=True)
# df['Ticket_freq'] = df['Ticket'].map(freq)

<a id="11"></a>
## 11. Escalado y normalización

Muchos algoritmos (KNN, SVM, regresión logística, redes neuronales, PCA) son
sensibles a la **escala** de las variables. Hay que ponerlas en un rango comparable.

| Scaler | Qué hace | Cuándo |
|---|---|---|
| **StandardScaler** | media 0, desviación 1 | Por defecto; datos ~normales |
| **MinMaxScaler** | escala a `[0, 1]` | Cuando se necesita un rango acotado |
| **RobustScaler** | usa mediana e IQR | Cuando hay outliers |

> ⚠️ Igual que con la imputación: **`fit` solo en train**, luego `transform` en test.
> Los árboles de decisión / Random Forest / XGBoost **no** necesitan escalado.


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

cols_num = ['Age', 'Fare']

scaler = StandardScaler()
df[cols_num] = scaler.fit_transform(df[cols_num])

print('Media tras escalar (~0):', df[cols_num].mean().round(3).tolist())
print('Std tras escalar  (~1):', df[cols_num].std().round(3).tolist())

In [ ]:
# ILUSTRATIVO — alternativas
# MinMaxScaler().fit_transform(df[cols_num])    # a rango [0, 1]
# RobustScaler().fit_transform(df[cols_num])    # robusto frente a outliers

<a id="12"></a>
## 12. Transformación de distribuciones

Variables muy **asimétricas** (como `Fare`) pueden mejorar el rendimiento de
modelos lineales si se transforman para acercarlas a una normal.

- **`log1p`** → `log(1+x)`, válido con ceros, reduce la cola derecha.
- **`sqrt`** → asimetría moderada.
- **Power / Box-Cox / Yeo-Johnson** → busca automáticamente la mejor transformación.


In [ ]:
# Comparación antes/después de aplicar log a Fare (usamos el original sin escalar)
fare = df_train['Fare'].fillna(df_train['Fare'].median())

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].hist(fare, bins=30); axes[0].set_title('Fare original (asimétrica)')
axes[1].hist(np.log1p(fare), bins=30); axes[1].set_title('log1p(Fare) (más simétrica)')
plt.tight_layout(); plt.show()

In [ ]:
# ILUSTRATIVO — PowerTransformer (Yeo-Johnson admite ceros y negativos)
# from sklearn.preprocessing import PowerTransformer
# pt = PowerTransformer(method='yeo-johnson')
# df[['Fare']] = pt.fit_transform(df[['Fare']])

<a id="13"></a>
## 13. Feature engineering (creación de variables)

A menudo el mayor salto de rendimiento viene de **crear variables nuevas** a partir
de las existentes, aportando conocimiento del dominio.


In [ ]:
# --- 13.1 Combinar variables: tamaño de la familia ---
# SibSp (hermanos/cónyuge) + Parch (padres/hijos) + 1 (la propia persona)
df_train['FamilySize'] = df_train['SibSp'] + df_train['Parch'] + 1
df_train['IsAlone'] = (df_train['FamilySize'] == 1).astype(int)
df_train[['SibSp', 'Parch', 'FamilySize', 'IsAlone']].head()

In [ ]:
# --- 13.2 Binning: convertir una numérica en categorías (rangos) ---
# pd.cut → cortes en rangos fijos
df_train['GrupoEdad'] = pd.cut(
    df_train['Age'],
    bins=[0, 12, 18, 40, 60, 100],
    labels=['Niño', 'Adolescente', 'Adulto', 'Maduro', 'Mayor']
)
df_train['GrupoEdad'].value_counts()

In [ ]:
# pd.qcut → cortes por cuantiles (mismo nº de elementos en cada grupo)
df_train['GrupoTarifa'] = pd.qcut(df_train['Fare'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
df_train['GrupoTarifa'].value_counts()

In [ ]:
# ILUSTRATIVO — extraer componentes de una fecha
# df['anio']     = df['fecha'].dt.year
# df['mes']      = df['fecha'].dt.month
# df['dia_sem']  = df['fecha'].dt.dayofweek      # 0 = lunes
# df['es_finde'] = df['fecha'].dt.dayofweek.isin([5, 6]).astype(int)
# df['antiguedad_dias'] = (pd.Timestamp('today') - df['fecha']).dt.days

<a id="14"></a>
## 14. Selección de características

Eliminar variables irrelevantes o redundantes reduce el sobreajuste, acelera el
entrenamiento y mejora la interpretabilidad.


In [ ]:
# --- 14.1 Matriz de correlación (detectar variables redundantes) ---
num = df_train.select_dtypes(include=np.number)
plt.figure(figsize=(8, 6))
sns.heatmap(num.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de correlación')
plt.show()

In [ ]:
# ILUSTRATIVO — métodos automáticos de selección
# from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif

# 1) Eliminar variables casi constantes (poca varianza)
# VarianceThreshold(threshold=0.0).fit_transform(X)

# 2) Quedarse con las K mejores según un test estadístico
# SelectKBest(score_func=f_classif, k=5).fit_transform(X, y)

# 3) Importancia según un modelo de árboles
# from sklearn.ensemble import RandomForestClassifier
# rf = RandomForestClassifier().fit(X, y)
# pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

<a id="15"></a>
## 15. Datos desbalanceados

Cuando una clase es mucho más frecuente que otra (fraude, enfermedades raras…), el
modelo tiende a ignorar la minoritaria. Estrategias:

- **`class_weight='balanced'`** en el modelo (lo más simple).
- **Oversampling** de la minoritaria (p.ej. **SMOTE**).
- **Undersampling** de la mayoritaria.


In [ ]:
# Comprobar el balance de la variable objetivo
df_train['Survived'].value_counts(normalize=True).round(3)

In [ ]:
# ILUSTRATIVO — SMOTE (requiere imbalanced-learn: %pip install imbalanced-learn)
# from imblearn.over_sampling import SMOTE
# X_res, y_res = SMOTE(random_state=42).fit_resample(X_train, y_train)

# Alternativa sin librerías extra: pasar class_weight al modelo
# from sklearn.linear_model import LogisticRegression
# LogisticRegression(class_weight='balanced')

<a id="16"></a>
## 16. División train / test

Se separa **antes** de ajustar imputadores/escaladores para evitar *data leakage*.
`stratify=y` mantiene la misma proporción de clases en ambos conjuntos.


In [ ]:
from sklearn.model_selection import train_test_split

# Suponiendo X (features) e y (objetivo) ya preparados:
feats = ['Pclass', 'Sex', 'Age', 'Fare', 'FamilySize']
datos = df_train.dropna(subset=feats + ['Survived']).copy()
datos['Sex'] = datos['Sex'].map({'male': 0, 'female': 1})

X = datos[feats]
y = datos['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% para test
    random_state=42,      # reproducibilidad
    stratify=y            # mantener proporción de clases
)
print('Train:', X_train.shape, ' Test:', X_test.shape)

<a id="17"></a>
## 17. Pipelines y ColumnTransformer

La forma **profesional** de encadenar todo el preprocesamiento. Ventajas:

- Aplica los mismos pasos a train y test **sin fugas de datos**.
- Trata por separado columnas numéricas y categóricas con `ColumnTransformer`.
- Se guarda y se reutiliza en producción de una sola pieza.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Definimos qué columnas son numéricas y cuáles categóricas
num_cols = ['Age', 'Fare', 'SibSp', 'Parch']
cat_cols = ['Pclass', 'Sex', 'Embarked']

# Sub-pipeline para numéricas: imputar mediana + escalar
pipe_num = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

# Sub-pipeline para categóricas: imputar moda + one-hot
pipe_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first')),
])

# ColumnTransformer aplica cada sub-pipeline a sus columnas
preprocesador = ColumnTransformer([
    ('num', pipe_num, num_cols),
    ('cat', pipe_cat, cat_cols),
])

preprocesador

In [ ]:
# Ajustar el preprocesador con el train y transformar
X_full = df_train[num_cols + cat_cols]
X_proc = preprocesador.fit_transform(X_full)
print('Forma tras el preprocesamiento:', X_proc.shape)

# Y se puede acoplar a un modelo en un único pipeline:
# from sklearn.ensemble import RandomForestClassifier
# modelo = Pipeline([('prep', preprocesador),
#                    ('clf', RandomForestClassifier())])
# modelo.fit(X_train, y_train)

<a id="18"></a>
## 18. Guardado del dataset y del pipeline

Tras el preprocesamiento, se guarda el dataset limpio y —muy importante— el
**pipeline ajustado**, para aplicar exactamente las mismas transformaciones a
datos nuevos en producción.


In [ ]:
# Guardar el DataFrame limpio
df_train.to_csv('../data/titanic_guia_limpio.csv', index=False)
print('Dataset guardado.')

# ILUSTRATIVO — guardar el pipeline entrenado con joblib
# import joblib
# joblib.dump(preprocesador, '../data/preprocesador.joblib')
# # ...más tarde, en producción:
# # preprocesador = joblib.load('../data/preprocesador.joblib')
# # X_nuevos_proc = preprocesador.transform(X_nuevos)

<a id="19"></a>
## 19. ✅ Checklist final de preprocesamiento

1. **Cargar** los datos con los parámetros correctos (separador, encoding, fechas).
2. **Explorar**: dimensiones, tipos, estadísticos, cardinalidad.
3. **Nulos**: analizar el patrón → eliminar / imputar según el porcentaje.
4. **Duplicados**: detectar y eliminar.
5. **Tipos**: corregir (numérico, fecha, categórico).
6. **Texto**: limpiar y extraer información útil.
7. **Outliers**: detectar (IQR / z-score) y tratar (recortar / eliminar).
8. **Categóricas**: codificar (one-hot, ordinal, mapeo).
9. **Escalado**: normalizar las numéricas si el modelo lo requiere.
10. **Distribuciones**: transformar variables muy asimétricas.
11. **Feature engineering**: crear variables nuevas con sentido de negocio.
12. **Selección**: descartar variables redundantes o irrelevantes.
13. **Desbalanceo**: corregir si las clases están desequilibradas.
14. **Split** train/test **antes** de ajustar imputadores/escaladores.
15. **Pipeline**: encapsular todo y **guardarlo** para producción.

> 🔑 **Las dos reglas que nunca debes olvidar:**
> 1. **Ajusta (`fit`) solo con el train** y aplica (`transform`) al test → evita el *data leakage*.
> 2. **Documenta y guarda el pipeline** → preprocesamiento reproducible en producción.
